In [ ]:
4!pip install langgraph langsmith

In [ ]:
!pip install langchain langchain_groq langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
pip install requests==2.32.4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [ ]:
pip install requests==2.32.4


In [ ]:
from google.colab import userdata
groq_api_key = userdata.get('groq_api_key')

In [ ]:
langsmith = userdata.get('langsmith')

In [ ]:
import os
os.environ ["LANGCHAIN_API_KEY"]=langsmith
os.environ ["LANGCHAIN_TRACING_V2"]="true"
os.environ ["LANGCHAIN_PROJECT"]="CourseLangGraph"

from langchain_groq import ChatGroq
llm = ChatGroq(groq_api_key=groq_api_key, model_name = "Gemma2-9b-It")


##Start Building Chatbot using Langgraph

In [ ]:
!pip -q install langgraph langchain langchain-groq

from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from google.colab import userdata
from IPython.display import Image, display

# Get API Key
groq_api_key = userdata.get("groq_api_key")

# Initialize LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.3-70b-versatile"
)

# Define State
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Build Graph
graph_builder = StateGraph(State)

def chatbot(state: State):
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph_builder.add_node("Chatbot", chatbot)
graph_builder.add_edge(START, "Chatbot")
graph_builder.add_edge("Chatbot", END)

graph = graph_builder.compile()

# Display Graph (Optional)
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

# Chat Loop
while True:
    user_input = input("User: ")

    if user_input.lower() in ["quit", "q", "exit"]:
        print("Good Bye!")
        break

    for event in graph.stream({"messages": [("user", user_input)]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)